# Week 3 — build your chunks

This notebook turns your notes into the `chunks.json` file your site reads.

**Everyone** runs sections 1 to 4.
**Option B and above** also run sections 5 to 7.

Nothing here costs money and nothing needs an API key.

## 1. Get your notes into Colab

Run the cell below and pick your five markdown files from `content/`:
`about.md`, `experience.md`, `projects.md`, `skills.md`, `faq.md`.

If you have not written them yet, run it and press Cancel. It will write a
small sample set instead so you can see the whole notebook work, then you can
come back later with your own.

In [ ]:
import os, json, textwrap

os.makedirs("content", exist_ok=True)
NEEDED = ["about.md", "experience.md", "projects.md", "skills.md", "faq.md"]

uploaded = {}
try:
    from google.colab import files          # only exists inside Colab
    print("Pick your markdown files (Cancel to use the sample set).")
    uploaded = files.upload()
except Exception as e:
    print(f"Not running in Colab ({type(e).__name__}). Looking for local files.")

for name, data in uploaded.items():
    with open(os.path.join("content", name), "wb") as fh:
        fh.write(data)
    print(f"saved content/{name}")

missing = [n for n in NEEDED if not os.path.exists(os.path.join("content", n))]
if missing:
    print(f"\nMissing {missing}, writing a sample set so the notebook runs.")
    SAMPLE = {
        "about.md": "# About\n\nI am Sample Person, an AI engineer. I am based in "
                    "Bangalore, India, and I work remotely.\n",
        "experience.md": "# Experience\n\n## AI Engineer, Northwind Data\n\nI own the "
                    "retrieval layer of a support assistant. Splitting documents on "
                    "headings instead of a fixed character count was worth 14 points "
                    "of accuracy on its own.\n",
        "projects.md": "# Projects\n\n## News triage assistant\n\nSorts a news item "
                    "into one of four categories and records every run so the "
                    "behaviour can be scored.\n",
        "skills.md": "# Skills\n\n## Languages\n\nPython is my main language. "
                    "JavaScript second.\n",
        "faq.md": "# Frequently asked\n\n## What is your notice period?\n\n"
                    "Thirty days.\n",
    }
    for n in missing:
        with open(os.path.join("content", n), "w", encoding="utf-8") as fh:
            fh.write(SAMPLE[n])

print("\ncontent/ now holds:")
for n in sorted(os.listdir("content")):
    print(f"  {n:<16} {os.path.getsize(os.path.join('content', n))} bytes")

## 2. The chunker

Three strategies. Read them before you run them, because the choice you make
here changes the answers more than anything you will do to the prompt.

- `whole` — one chunk per file. Simple, and usually the worst.
- `heading` — split on `## ` headings. Follows the shape of the document.
- `window` — fixed character windows with overlap. Ignores structure entirely.

In [ ]:
import json
import os
import re
import sys
from datetime import datetime, timezone

CONTENT_DIR = "content"


def read_files(filenames):
    out = {}
    for name in filenames:
        path = os.path.join(CONTENT_DIR, name)
        with open(path, encoding="utf-8") as fh:
            out[name] = fh.read()
    return out


def normalise(text):
    """Collapse single newlines inside a paragraph, keep paragraph breaks."""
    text = text.replace("\r\n", "\n")
    paras = [re.sub(r"\s*\n\s*", " ", p).strip() for p in text.split("\n\n")]
    return "\n\n".join(p for p in paras if p)


def chunk_whole(name, text):
    body = normalise(re.sub(r"^#\s+.*\n", "", text, count=1))
    return [{"file": name, "heading": name.replace(".md", ""), "text": body}]


def chunk_heading(name, text):
    """Split on '## '. Text before the first '## ' is kept as its own chunk."""
    text = re.sub(r"^#\s+.*\n", "", text, count=1)
    parts = re.split(r"^##\s+(.+)$", text, flags=re.MULTILINE)
    chunks = []
    preamble = normalise(parts[0])
    if preamble:
        chunks.append({"file": name, "heading": name.replace(".md", ""),
                       "text": preamble})
    for i in range(1, len(parts), 2):
        heading = parts[i].strip()
        body = normalise(parts[i + 1]) if i + 1 < len(parts) else ""
        if body:
            chunks.append({"file": name, "heading": heading,
                           "text": heading + ". " + body})
    return chunks


def chunk_window(name, text, size=600, overlap=150):
    body = normalise(re.sub(r"^#\s+.*\n", "", text, count=1))
    flat = body.replace("\n\n", " ")
    step = size - overlap
    if step <= 0:
        raise ValueError("overlap must be smaller than size")
    chunks = []
    for start in range(0, max(len(flat), 1), step):
        piece = flat[start:start + size].strip()
        if len(piece) < 80:
            continue
        chunks.append({"file": name, "heading": name.replace(".md", ""),
                       "text": piece})
        if start + size >= len(flat):
            break
    return chunks


STRATEGIES = {"whole": chunk_whole, "heading": chunk_heading,
              "window": chunk_window}


def build(filenames, strategy="heading"):
    if strategy not in STRATEGIES:
        raise ValueError(f"unknown strategy {strategy!r}; "
                         f"pick one of {sorted(STRATEGIES)}")
    fn = STRATEGIES[strategy]
    chunks, counter = [], {}
    for name, text in read_files(filenames).items():
        for ch in fn(name, text):
            stem = name.replace(".md", "")
            counter[stem] = counter.get(stem, 0) + 1
            ch["id"] = f"{stem}-{counter[stem]}"
            ch["chars"] = len(ch["text"])
            chunks.append(ch)
    return {
        "strategy": strategy,
        "built_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "files": list(filenames),
        "chunks": chunks,
    }


def report(doc):
    chunks = doc["chunks"]
    sizes = [c["chars"] for c in chunks]
    print(f"strategy   : {doc['strategy']}")
    print(f"files      : {len(doc['files'])}")
    print(f"chunks     : {len(chunks)}")
    if sizes:
        print(f"chars      : min {min(sizes)}, mean {sum(sizes)//len(sizes)}, "
              f"max {max(sizes)}")
    by_file = {}
    for c in chunks:
        by_file[c["file"]] = by_file.get(c["file"], 0) + 1
    for name in doc["files"]:
        print(f"  {name:<18} {by_file.get(name, 0)} chunks")


## 3. Try all three and look at the difference

This prints the shape of each strategy. There is no single right answer, but
there is a wrong one: whichever produces chunks so large that every answer
drags in text about something else.

In [ ]:
FILES = ["about.md", "experience.md", "projects.md", "skills.md"]

for name in STRATEGIES:
    doc = build(FILES, name)
    print("=" * 60)
    report(doc)
    print()

## 4. Write chunks.json

Change `STRATEGY` below if you want a different one, then run it. The last
line downloads the file. Put it in the root of your repository, next to
`index.html`, and commit it.

**Note which files are in the list.** `faq.md` is deliberately left out, to
match `config.js`. Three of the questions in `questions.json` can only be
answered from it. Your Option A task is to find that out by measurement, not
by reading this comment.

In [ ]:
STRATEGY = "heading"
FILES = ["about.md", "experience.md", "projects.md", "skills.md"]

doc = build(FILES, STRATEGY)
report(doc)

with open("chunks.json", "w", encoding="utf-8") as fh:
    json.dump(doc, fh, indent=1, ensure_ascii=False)
print("\nwrote chunks.json")

try:
    from google.colab import files
    files.download("chunks.json")
except Exception:
    print("(not in Colab, so no download; the file is in this folder)")

---

# Option B and above starts here

Everything above produced chunks of text. Everything below turns them into
vectors and then measures whether that actually helped.

## 5. Fit a TF-IDF vectoriser

This is the same `TfidfVectorizer` you used in Week 1 for the news classifier.
The difference is that this time you keep the vocabulary and the IDF weights,
and export them, so that JavaScript in the browser can build a vector for the
visitor's question using the same vocabulary.

**`stop_words` is the setting that matters most.** scikit-learn removes no
stopwords by default. Leave them in on a corpus this small and the question
"Where are you based?" spends most of its vector on "where", "are" and "you",
which appear in nearly every chunk. Worse, nonsense questions start matching
things, because "the" matches everywhere, and your refusal stops working.
Try it both ways in section 6 and you will see it.

In [ ]:
import json
import sys

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer


def vectorise(doc, min_df=1, sublinear_tf=True, stop_words="english"):
    texts = [c["text"] for c in doc["chunks"]]

    # stop_words matters more than anything else here. scikit-learn removes no
    # stopwords by default, and on a corpus this small IDF cannot tell "are"
    # apart from "based" (2.00 against 2.85). Leave them in and the question
    # "Where are you based?" spends 60% of its vector on "where", "are", "you".
    vec = TfidfVectorizer(
        lowercase=True,
        sublinear_tf=sublinear_tf,
        smooth_idf=True,
        norm="l2",
        min_df=min_df,
        stop_words=stop_words,
        token_pattern=r"(?u)\b\w\w+\b",
    )
    matrix = vec.fit_transform(texts)

    vocab = {term: int(i) for term, i in vec.vocabulary_.items()}
    idf = [float(x) for x in vec.idf_]

    out = dict(doc)
    out["vectoriser"] = {
        "kind": "tfidf",
        "stop_words": stop_words,
        "sublinear_tf": sublinear_tf,
        "smooth_idf": True,
        "norm": "l2",
        "token_pattern": r"\b\w\w+\b",
        "vocab_size": len(vocab),
        "vocab": vocab,
        "idf": idf,
    }

    chunks = []
    for c, row in zip(doc["chunks"], matrix):
        coo = row.tocoo()
        chunks.append({
            **c,
            "vec": {str(int(j)): round(float(v), 7)
                    for j, v in zip(coo.col, coo.data)},
        })
    out["chunks"] = chunks
    return out, vec, matrix


In [ ]:
doc = build(["about.md", "experience.md", "projects.md", "skills.md", "faq.md"],
            "heading")

out, vec, matrix = vectorise(doc, stop_words="english")
print(f"chunks     : {len(out['chunks'])}")
print(f"vocabulary : {out['vectoriser']['vocab_size']} terms")
print(f"idf range  : {min(out['vectoriser']['idf']):.4f} "
      f"to {max(out['vectoriser']['idf']):.4f}")

with open("chunks.json", "w", encoding="utf-8") as fh:
    json.dump(out, fh, indent=1, ensure_ascii=False)
print("\nwrote chunks.json with vectors")

try:
    from google.colab import files
    files.download("chunks.json")
except Exception:
    print("(not in Colab, so no download)")

## 6. Prove whether it helped

Retrieval offline, in Python, so you can compare methods without deploying
anything. Three numbers per method:

- **found** — did any returned chunk contain the answer at all
- **MRR** — how high up the ranking it was. 1.0 means always first
- **waste** — what fraction of the characters you are about to pay for sit in
  chunks that cannot answer the question

Run it with `stop_words=None` as well. Watch the last two questions, the ones
that should be refused, start getting answers.

In [ ]:
import numpy as np

QUESTIONS = [
    ("Where are you based?",                     ["bangalore"]),
    ("What is your main programming language?",   ["python"]),
    ("What is your notice period?",               ["thirty days"]),
    ("How soon could you start?",                 ["thirty days"]),
    ("What city do you live in?",                 ["bangalore"]),
    ("Who won the cricket world cup?",            []),
    ("What is the capital of Brazil?",            []),
]
PASSAGES, FLOOR = 3, 0.05


def rank(question, out, vec, matrix):
    q = vec.transform([question])
    sims = (matrix @ q.T).toarray().ravel()
    order = np.argsort(-sims)[:PASSAGES]
    return [(out["chunks"][i], float(sims[i])) for i in order
            if sims[i] > FLOOR]


def measure(stop_words):
    out, vec, matrix = vectorise(doc, stop_words=stop_words)
    found = rr = waste_sum = waste_n = 0
    print(f"  stop_words={stop_words!r}  vocab={out['vectoriser']['vocab_size']}")
    for q, expect in QUESTIONS:
        hits = rank(q, out, vec, matrix)
        if not expect:
            ok = len(hits) == 0
            found += ok
            rr += ok
            print(f"    {'ok  ' if ok else 'FAIL'} refuse  {q}")
            continue
        texts = [c["text"].lower() for c, _ in hits]
        pos = next((i for i, t in enumerate(texts)
                    if any(e in t for e in expect)), None)
        found += pos is not None
        rr += 0 if pos is None else 1 / (pos + 1)
        total = sum(len(c["text"]) for c, _ in hits)
        if total:
            useful = sum(len(c["text"]) for c, _ in hits
                         if any(e in c["text"].lower() for e in expect))
            waste_sum += (total - useful) / total
            waste_n += 1
        where = "nothing" if pos is None else f"rank {pos + 1}"
        print(f"    {'ok  ' if pos == 0 else 'ok  ' if pos is not None else 'FAIL'}"
              f" {where:<9} {q}")
    n = len(QUESTIONS)
    print(f"    found {found}/{n}   MRR {rr / n:.3f}   "
          f"waste {waste_sum / max(waste_n, 1) * 100:.1f}%\n")


print("WITH stopword removal")
measure("english")
print("WITHOUT stopword removal")
measure(None)

## 7. Optional: dense embeddings

Everything so far counts words. That has a hard ceiling, and section 6 shows
you where it is: "How soon could you start?" shares no word at all with "My
notice period is thirty days", so no amount of weighting will connect them.

Dense embeddings are the fix. The cell below computes them with
`sentence-transformers`, which downloads a model of roughly 90 MB the first
time and then runs on Colab's CPU in a few seconds.

**Two honest warnings.**

First, this is an offline comparison only. Serving dense embeddings from your
site needs the same model in the browser, which is a bigger change than this
assignment asks for. Measure the gain here, write it up, and ship the TF-IDF
version.

Second, I have not been able to run this cell in the environment where the
rest of this notebook was tested, because that machine has no access to the
model host. Everything above has verified numbers. This cell does not. If it
fails, that is worth reporting, and it does not affect your submission.

In [ ]:
# !pip install -q sentence-transformers

try:
    from sentence_transformers import SentenceTransformer
    import numpy as np

    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    texts = [c["text"] for c in doc["chunks"]]
    M = model.encode(texts, normalize_embeddings=True)
    print(f"embedded {M.shape[0]} chunks into {M.shape[1]} dimensions")

    found = rr = 0
    for q, expect in QUESTIONS:
        qv = model.encode([q], normalize_embeddings=True)
        sims = (M @ qv.T).ravel()
        order = np.argsort(-sims)[:PASSAGES]
        hits = [(doc["chunks"][i], float(sims[i])) for i in order
                if sims[i] > 0.15]
        if not expect:
            ok = len(hits) == 0
            found += ok; rr += ok
            print(f"  {'ok  ' if ok else 'FAIL'} refuse  {q}")
            continue
        pos = next((i for i, (c, _) in enumerate(hits)
                    if any(e in c["text"].lower() for e in expect)), None)
        found += pos is not None
        rr += 0 if pos is None else 1 / (pos + 1)
        print(f"  {'ok  ' if pos is not None else 'FAIL'} "
              f"{'nothing' if pos is None else 'rank ' + str(pos + 1):<9} {q}")
    n = len(QUESTIONS)
    print(f"\n  dense: found {found}/{n}   MRR {rr / n:.3f}")
    print("  Compare against the TF-IDF numbers from section 6.")
except Exception as e:
    print(f"Dense embeddings did not run: {type(e).__name__}: {e}")
    print("Report this and move on. Your submission does not depend on it.")